# Notebook 3 — Transformer Fine-Tuning (Main Model)
**Member 3 Responsibility**

This is the **core model** of the project. We fine-tune `roberta-base` as a regression model to predict plausibility scores.

## Architecture
- Input: `[CLS] precontext + sentence + ending [SEP] judged_meaning [SEP]`
- Model: RoBERTa-base → linear regression head → predict score in [1, 5]
- Loss: MSE (regression task)

## Requirements
```
pip install transformers torch datasets
```

**Note:** Training takes ~10–20 min on CPU, ~3–5 min with GPU.
The model is saved locally — **no API key needed** (satisfies the course requirement).

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

with open('train.json') as f:
    train_raw = json.load(f)
with open('dev.json') as f:
    dev_raw = json.load(f)

def to_df(raw):
    rows = []
    for sid, s in raw.items():
        rows.append({
            'id': sid,
            'homonym': s['homonym'],
            'judged_meaning': s['judged_meaning'],
            'precontext': s.get('precontext', ''),
            'sentence': s['sentence'],
            'ending': s.get('ending', '') or '',
            'average': s['average'],
            'stdev': s['stdev'],
            'example_sentence': s.get('example_sentence', '') or ''
        })
    return pd.DataFrame(rows)

train_df = to_df(train_raw)
dev_df   = to_df(dev_raw)
print(f'Train: {len(train_df)} | Dev: {len(dev_df)}')

## 1. Dataset Class

In [ ]:
MODEL_NAME  = 'roberta-base'  # downloads ~500MB once, cached locally after that
MAX_LEN     = 256
BATCH_SIZE  = 16
NUM_EPOCHS  = 4
LR          = 2e-5

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

class AmbiStoryDataset(Dataset):
    """
    PyTorch Dataset for the AmbiStory task.
    
    Each sample is encoded as:
      [CLS] story_context [SEP][SEP] judged_meaning [SEP]
    where story_context = precontext + ambiguous sentence + optional ending.
    """
    def __init__(self, df, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build the two text segments for the pair encoding
        parts = [row['precontext'], row['sentence']]
        if row['ending']:
            parts.append(row['ending'])
        story = ' '.join(parts)
        meaning = row['judged_meaning']

        encoding = self.tokenizer(
            story,
            meaning,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(row['average'], dtype=torch.float),
            'stdev':          torch.tensor(row['stdev'],   dtype=torch.float),
            'id':             str(row['id'])
        }

train_dataset = AmbiStoryDataset(train_df, tokenizer, MAX_LEN)
dev_dataset   = AmbiStoryDataset(dev_df,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'DataLoaders ready. Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}')

## 2. Model Architecture

In [ ]:
class PlausibilityRegressor(nn.Module):
    """
    RoBERTa with a regression head.
    Predicts a plausibility score in [1, 5].
    """
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.roberta  = RobertaModel.from_pretrained(model_name)
        hidden_size   = self.roberta.config.hidden_size  # 768 for roberta-base
        self.dropout  = nn.Dropout(dropout)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs  = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        cls_repr = self.dropout(cls_repr)
        score    = self.regressor(cls_repr).squeeze(-1)  # (batch,)
        # Sigmoid-scale output to [1, 5]
        score    = 1 + 4 * torch.sigmoid(score)
        return score

model = PlausibilityRegressor(MODEL_NAME).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Parameters: {total_params:,}')

## 3. Training

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
loss_fn = nn.MSELoss()

def evaluate_model(model, loader, df):
    """Run model on a dataloader, return official metrics."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            masks = batch['attention_mask'].to(device)
            preds = model(ids, masks).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(batch['label'].numpy().tolist())
    preds_arr = np.clip(np.array(all_preds), 1, 5)
    trues_arr = np.array(all_labels)
    stdevs    = df['stdev'].values
    rho, _    = spearmanr(preds_arr, trues_arr)
    margin    = np.maximum(stdevs, 1.0)
    acc       = (np.abs(preds_arr - trues_arr) <= margin).mean()
    return {'spearman_r': round(float(rho), 4), 'acc_within_stdev': round(float(acc), 4)}, preds_arr

train_losses = []
dev_spearman = []
best_spearman = -1.0

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids   = batch['input_ids'].to(device)
        attn_mask   = batch['attention_mask'].to(device)
        labels      = batch['label'].to(device)

        optimizer.zero_grad()
        preds = model(input_ids, attn_mask)
        loss  = loss_fn(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        if (step + 1) % 30 == 0:
            print(f'  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    dev_metrics, dev_preds = evaluate_model(model, dev_loader, dev_df)
    dev_spearman.append(dev_metrics['spearman_r'])
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS} — Train Loss: {avg_loss:.4f} | Dev: {dev_metrics}')

    # Save best model
    if dev_metrics['spearman_r'] > best_spearman:
        best_spearman = dev_metrics['spearman_r']
        torch.save(model.state_dict(), 'best_roberta_model.pt')
        best_dev_preds = dev_preds.copy()
        best_dev_metrics = dev_metrics
        print(f'  ✓ New best model saved (Spearman={best_spearman:.4f})')

print('\nTraining complete!')
print('Best dev metrics:', best_dev_metrics)

## 4. Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, NUM_EPOCHS+1), train_losses, marker='o', color='steelblue')
axes[0].set_title('Training Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')

axes[1].plot(range(1, NUM_EPOCHS+1), dev_spearman, marker='o', color='coral')
axes[1].set_title('Dev Spearman Correlation per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curve.png')

## 5. Error Analysis

In [ ]:
dev_df_copy = dev_df.copy()
dev_df_copy['pred'] = best_dev_preds
dev_df_copy['error'] = np.abs(dev_df_copy['pred'] - dev_df_copy['average'])

print('=== Error Analysis ===')
print(f'Mean Absolute Error: {dev_df_copy["error"].mean():.3f}')
print(f'Median Abs Error:    {dev_df_copy["error"].median():.3f}')

# Worst predictions
print('\n--- Top 5 Worst Predictions ---')
worst = dev_df_copy.nlargest(5, 'error')[['homonym','sentence','judged_meaning','average','pred','error']]
for _, r in worst.iterrows():
    print(f'  homonym={r["homonym"]} | true={r["average"]:.1f} | pred={r["pred"]:.2f} | err={r["error"]:.2f}')
    print(f'  sentence: {r["sentence"][:80]}')
    print(f'  meaning:  {r["judged_meaning"][:80]}')
    print()

# Scatter plot of predictions vs true scores
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(dev_df_copy['average'], dev_df_copy['pred'], alpha=0.4, s=15, color='steelblue')
ax.plot([1, 5], [1, 5], 'r--', linewidth=1)
ax.set_xlabel('True Score')
ax.set_ylabel('Predicted Score')
ax.set_title(f'Predictions vs True Scores (Dev)\nSpearman ρ={best_dev_metrics["spearman_r"]}')
plt.tight_layout()
plt.savefig('prediction_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prediction_scatter.png')

## 6. Save Results

In [ ]:
transformer_results = {'RoBERTa-base (fine-tuned)': best_dev_metrics}
with open('transformer_results.json', 'w') as f:
    json.dump(transformer_results, f, indent=2)
print('Saved: transformer_results.json')
print('Best model weights: best_roberta_model.pt')
print('\nFinal results:', best_dev_metrics)

## ✅ Notebook 3 Complete
Files produced:
- `best_roberta_model.pt` — fine-tuned model weights
- `transformer_results.json` — dev evaluation results
- `training_curve.png`
- `prediction_scatter.png`

Next: Run **notebook4_report_and_predict.ipynb** — generates `predict.py` output and the final results table.